# 🎙️ FastPitch + HiFiGAN — Multi-Speaker TTS Inference
**NVIDIA DeepLearningExamples FastPitch · NeMo & VCTK HiFiGAN vocoders**

---
### One-time terminal setup
```bash
conda create -n nemo-tts python=3.10 -y
conda activate nemo-tts
pip install "numpy==1.26.4"
pip install "torch==2.6.0" "torchaudio==2.6.0" --index-url https://download.pytorch.org/whl/cpu
pip install "nemo_toolkit[tts]==2.7.0"
pip install soundfile librosa matplotlib ipywidgets ipykernel jupyter
python -m ipykernel install --user --name nemo-tts --display-name "Python 3 (nemo-tts)"
jupyter notebook
```
> ⚠️ Select the **`nemo-tts`** kernel (top-right corner) before running.

---
### Notebook flow
| Step | What it does | Run once? |
|------|-------------|----------|
| 1 | Verify environment | ✅ Once |
| 2 | Clone DLE + load FastPitch | ✅ Once per session |
| 3a | Load NeMo HiFiGAN vocoder | ✅ Once per session |
| 3b | Load VCTK HiFiGAN vocoder | ✅ Once per session |
| 4 | Define `build_tts_ui()` factory | ✅ Once per session |
| 5a | **Synthesize** — NeMo HiFiGAN UI | 🔁 Anytime |
| 5b | **Synthesize** — VCTK HiFiGAN UI | 🔁 Anytime |
| 6 | Batch render all speakers | 🔁 Optional |

---
## ✅ Step 1 — Verify Environment

In [9]:
import sys, os
import numpy as np
import torch

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

# Device: MPS for Apple Silicon, CUDA for NVIDIA, else CPU
device = "mps"  if torch.backends.mps.is_available() else \
         "cuda" if torch.cuda.is_available()         else "cpu"
print(f"Device  : {device.upper()}")

assert sys.version_info[:2] == (3, 10), "❌ Wrong Python — activate the nemo-tts conda env!"
assert int(np.__version__.split('.')[0]) < 2, "❌ NumPy 2.x — run: pip install 'numpy==1.26.4'"
print("\n✅ Environment looks good!")

Python  : 3.10.20
NumPy   : 1.26.4
PyTorch : 2.6.0
CUDA    : False
Device  : MPS

✅ Environment looks good!


---
## 📦 Step 2 — Load FastPitch Checkpoint
Set `FASTPITCH_CHECKPOINT` to your `.pt` file path, then run this cell.

In [10]:
# ✏️ ← Change this to your checkpoint path
FASTPITCH_CHECKPOINT = "/Users/ibrahimkhaliloglu/Downloads/FastPitch_checkpoint_3.pt"
OUTPUT_DIR = "./tts_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(FASTPITCH_CHECKPOINT), f"❌ File not found: {FASTPITCH_CHECKPOINT}"
print(f"✅ Found: {FASTPITCH_CHECKPOINT}  ({os.path.getsize(FASTPITCH_CHECKPOINT)/1e6:.1f} MB)")
print(f"   Outputs → {os.path.abspath(OUTPUT_DIR)}")

# ── Clone DLE (skipped if already present) ────────────────────────────────────
DLE_PATH = "/tmp/DLE/PyTorch/SpeechSynthesis/FastPitch"
if not os.path.exists(DLE_PATH):
    print("\nCloning NVIDIA DeepLearningExamples (one-time, ~1 min)...")
    os.system("git clone --quiet --depth 1 https://github.com/NVIDIA/DeepLearningExamples.git /tmp/DLE")
    print("Done.")
else:
    print("\n✅ DLE already cloned.")

if DLE_PATH not in sys.path:
    sys.path.insert(0, DLE_PATH)

# ── Load checkpoint ────────────────────────────────────────────────────────────
print("\nLoading checkpoint...", end=" ", flush=True)
ckpt = torch.load(FASTPITCH_CHECKPOINT, map_location="cpu")
cfg  = ckpt["config"]
print("done.")

# ── Build model directly from the embedded config ─────────────────────────────
from fastpitch.model import FastPitch

fp_model = FastPitch(
    n_mel_channels               = cfg["n_mel_channels"],
    n_symbols                    = cfg["n_symbols"],
    padding_idx                  = cfg["padding_idx"],
    symbols_embedding_dim        = cfg["symbols_embedding_dim"],
    in_fft_n_layers              = cfg["in_fft_n_layers"],
    in_fft_n_heads               = cfg["in_fft_n_heads"],
    in_fft_d_head                = cfg["in_fft_d_head"],
    in_fft_conv1d_kernel_size    = cfg["in_fft_conv1d_kernel_size"],
    in_fft_conv1d_filter_size    = cfg["in_fft_conv1d_filter_size"],
    in_fft_output_size           = cfg["in_fft_output_size"],
    p_in_fft_dropout             = cfg["p_in_fft_dropout"],
    p_in_fft_dropatt             = cfg["p_in_fft_dropatt"],
    p_in_fft_dropemb             = cfg["p_in_fft_dropemb"],
    out_fft_n_layers             = cfg["out_fft_n_layers"],
    out_fft_n_heads              = cfg["out_fft_n_heads"],
    out_fft_d_head               = cfg["out_fft_d_head"],
    out_fft_conv1d_kernel_size   = cfg["out_fft_conv1d_kernel_size"],
    out_fft_conv1d_filter_size   = cfg["out_fft_conv1d_filter_size"],
    out_fft_output_size          = cfg["out_fft_output_size"],
    p_out_fft_dropout            = cfg["p_out_fft_dropout"],
    p_out_fft_dropatt            = cfg["p_out_fft_dropatt"],
    p_out_fft_dropemb            = cfg["p_out_fft_dropemb"],
    dur_predictor_kernel_size    = cfg["dur_predictor_kernel_size"],
    dur_predictor_filter_size    = cfg["dur_predictor_filter_size"],
    p_dur_predictor_dropout      = cfg["p_dur_predictor_dropout"],
    dur_predictor_n_layers       = cfg["dur_predictor_n_layers"],
    pitch_predictor_kernel_size  = cfg["pitch_predictor_kernel_size"],
    pitch_predictor_filter_size  = cfg["pitch_predictor_filter_size"],
    p_pitch_predictor_dropout    = cfg["p_pitch_predictor_dropout"],
    pitch_predictor_n_layers     = cfg["pitch_predictor_n_layers"],
    pitch_embedding_kernel_size  = cfg["pitch_embedding_kernel_size"],
    n_speakers                   = cfg["n_speakers"],
    speaker_emb_weight           = cfg["speaker_emb_weight"],
    energy_predictor_kernel_size = cfg["energy_predictor_kernel_size"],
    energy_predictor_filter_size = cfg["energy_predictor_filter_size"],
    p_energy_predictor_dropout   = cfg["p_energy_predictor_dropout"],
    energy_predictor_n_layers    = cfg["energy_predictor_n_layers"],
    energy_conditioning          = cfg["energy_conditioning"],
    energy_embedding_kernel_size = cfg["energy_embedding_kernel_size"],
)

# Strip DataParallel 'module.' prefix and load weights
cleaned = {k.replace("module.", "", 1): v for k, v in ckpt["state_dict"].items()}
missing, unexpected = fp_model.load_state_dict(cleaned, strict=True)
fp_model = fp_model.eval().to(device)

N_SPEAKERS = cfg["n_speakers"]
print(f"\n✅ FastPitch loaded on {device.upper()}")
print(f"   Speakers : {N_SPEAKERS}  (IDs 0–{N_SPEAKERS - 1})")
print(f"   Missing  : {len(missing)}  |  Unexpected : {len(unexpected)}")

✅ Found: /Users/ibrahimkhaliloglu/Downloads/FastPitch_checkpoint_3.pt  (555.4 MB)
   Outputs → /Users/ibrahimkhaliloglu/Desktop/tts_output

✅ DLE already cloned.

Loading checkpoint... done.

✅ FastPitch loaded on MPS
   Speakers : 10  (IDs 0–9)
   Missing  : 0  |  Unexpected : 0


---
## 🔊 Step 3a — Load NeMo HiFiGAN Vocoder (tts_en_hifigan)

In [11]:
from nemo.collections.tts.models import HifiGanModel

HIFIGAN_MODEL = "tts_en_hifigan"
print(f"Loading {HIFIGAN_MODEL}  (downloads once, then cached)...")
vocoder_nemo = HifiGanModel.from_pretrained(HIFIGAN_MODEL).eval().to(device)
print(f"✅ NeMo HiFiGAN loaded on {device.upper()}")


Loading tts_en_hifigan  (downloads once, then cached)...
[NeMo I 2026-03-18 21:22:08 common:935] Found existing object /Users/ibrahimkhaliloglu/.cache/torch/NeMo/NeMo_2.7.0/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo.
[NeMo I 2026-03-18 21:22:08 common:935] Re-using file from: /Users/ibrahimkhaliloglu/.cache/torch/NeMo/NeMo_2.7.0/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo
[NeMo I 2026-03-18 21:22:08 common:869] Instantiating model from pre-trained checkpoint


[NeMo W 2026-03-18 21:22:12 hifigan:54] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/train_finetune.txt
      min_duration: 0.75
      n_segments: 8192
    dataloader_params:
      drop_last: false
      shuffle: true
      batch_size: 64
      num_workers: 4
    
[NeMo W 2026-03-18 21:22:12 hifigan:54] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    dataset:
      _target_: nemo.collections.tts.data.datalayers.MelAudioDataset
      manifest_filepath: /home/fkreuk/data/val_finetune.txt
      min_duration: 3
      n_segments: 66150
  

[NeMo I 2026-03-18 21:22:12 modelPT:501] Model HifiGanModel was successfully restored from /Users/ibrahimkhaliloglu/.cache/torch/NeMo/NeMo_2.7.0/tts_hifigan/e6da322f0f7e7dcf3f1900a9229a7e69/tts_hifigan.nemo.
✅ NeMo HiFiGAN loaded on MPS


---
## 🔊 Step 3b — Load HiFiGAN VCTK v1 (standalone)

In [12]:
import sys, json as _json

HIFIGAN_REPO = "/Users/ibrahimkhaliloglu/Desktop/hifi-gan"
if HIFIGAN_REPO not in sys.path:
    sys.path.insert(0, HIFIGAN_REPO)

from models import Generator
from env import AttrDict

VCTK_CONFIG = "/Users/ibrahimkhaliloglu/Desktop/hifi-gan/checkpoints/config.json"
VCTK_CKPT   = "/Users/ibrahimkhaliloglu/Desktop/hifi-gan/checkpoints/generator_v1"

with open(VCTK_CONFIG) as f:
    _h = AttrDict(_json.load(f))

vocoder_vctk = Generator(_h).to(device)
vocoder_vctk.load_state_dict(
    torch.load(VCTK_CKPT, map_location=device)["generator"]
)
vocoder_vctk.eval()
vocoder_vctk.remove_weight_norm()
print(f"✅ HiFiGAN VCTK v1 loaded on {device.upper()}")


Removing weight norm...
✅ HiFiGAN VCTK v1 loaded on MPS


---
## 🛠️ Step 4 — Define TTS UI Factory
Run once. Each call to `build_tts_ui()` creates a fully isolated widget UI — all widgets, callbacks and tokenizers live inside the function closure, so both UIs are completely independent.

In [13]:
import os, torch, soundfile as sf
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Audio, clear_output
from common.text.text_processing import TextProcessing

SAMPLE_RATE = 22050


def build_tts_ui(fp_model, vocoder, vocoder_type, device, n_speakers, output_dir):
    """
    Build a fully isolated TTS widget UI.

    Parameters
    ----------
    fp_model     : loaded FastPitch model (shared, read-only)
    vocoder      : NeMo HifiGanModel  OR  standalone hifi-gan Generator
    vocoder_type : "nemo" | "vctk"
    device       : torch device string  (e.g. "mps", "cpu")
    n_speakers   : number of speaker IDs available
    output_dir   : directory for saved .wav files
    """
    # Private tokenizer — never shared with other UIs
    _tp = TextProcessing(symbol_set="english_basic", cleaner_names=["english_cleaners"])

    def _tokenize(text):
        return torch.LongTensor(_tp.encode_text(text)).unsqueeze(0).to(device)

    def _run_vocoder(mel):
        if vocoder_type == "nemo":
            audio = vocoder.convert_spectrogram_to_audio(spec=mel)
            return audio[0].cpu().numpy()
        else:  # vctk — standalone Generator
            return vocoder(mel).squeeze().cpu().numpy()

    # ── Private widgets ───────────────────────────────────────────────────────
    _style  = {"description_width": "130px"}
    _layout = widgets.Layout(width="580px")

    _w_text = widgets.Textarea(
        value="Hello, this is a multi-speaker text to speech synthesis test.",
        description="📝 Text:",
        layout=widgets.Layout(width="580px", height="80px"),
        style=_style,
    )
    _w_speaker = widgets.IntSlider(
        value=0, min=0, max=n_speakers - 1, step=1,
        description=f"🎤 Speaker (0\u2013{n_speakers-1}):",
        continuous_update=False, layout=_layout, style=_style,
    )
    _w_pace = widgets.FloatSlider(
        value=1.0, min=0.5, max=2.0, step=0.05,
        description="⏩ Pace:",
        continuous_update=False, readout_format=".2f",
        layout=_layout, style=_style,
    )
    _w_pitch = widgets.FloatSlider(
        value=1.0, min=0.5, max=2.0, step=0.05,
        description="🎵 Pitch scale:",
        continuous_update=False, readout_format=".2f",
        layout=_layout, style=_style,
    )
    _w_show_mel = widgets.Checkbox(
        value=True, description="Show mel spectrogram",
        layout=widgets.Layout(width="210px"),
    )
    _w_save = widgets.Checkbox(
        value=True, description="Save .wav to output folder",
        layout=widgets.Layout(width="220px"),
    )
    _btn = widgets.Button(
        description=" \u25b6  Synthesize",
        button_style="success", icon="play",
        layout=widgets.Layout(width="180px", height="40px"),
    )
    _out = widgets.Output()

    # ── Private callback ──────────────────────────────────────────────────────
    def _synthesize(_event):
        with _out:
            clear_output(wait=True)
            text        = _w_text.value.strip()
            speaker_id  = _w_speaker.value
            pace        = _w_pace.value
            pitch_scale = _w_pitch.value

            if not text:
                print("\u26a0\ufe0f  Please enter some text first.")
                return

            print(f'\U0001f4dd "{text}"')
            print(f"\U0001f3a4 Speaker: {speaker_id}  |  "
                  f"\u23e9 Pace: {pace:.2f}  |  "
                  f"\U0001f3b5 Pitch: {pitch_scale:.2f}  |  "
                  f"\U0001f50a Vocoder: {vocoder_type}")
            print("Running inference...", end=" ", flush=True)

            try:
                tokens  = _tokenize(text)
                speaker = torch.tensor([speaker_id], dtype=torch.long, device=device)

                with torch.no_grad():
                    mel, *_ = fp_model.infer(
                        tokens,
                        speaker=speaker,
                        pace=pace,
                        pitch_tgt=None,
                        pitch_transform=lambda p, *a, **kw: p * pitch_scale,
                    )
                    audio_np = _run_vocoder(mel)

                print(f"done.  \u23f1 {len(audio_np)/SAMPLE_RATE:.2f}s")

                if _w_show_mel.value:
                    fig, ax = plt.subplots(figsize=(12, 3))
                    img = ax.imshow(
                        mel[0].cpu().numpy(),
                        aspect="auto", origin="lower", cmap="viridis"
                    )
                    ax.set_title(
                        f"Mel Spectrogram \u2014 Speaker {speaker_id} [{vocoder_type}]",
                        fontsize=12
                    )
                    ax.set_xlabel("Frames")
                    ax.set_ylabel("Mel Bins")
                    plt.colorbar(img, ax=ax)
                    plt.tight_layout()
                    plt.show()

                if _w_save.value:
                    fname = (
                        f"{vocoder_type}_spk{speaker_id}"
                        f"_pace{pace:.2f}_pitch{pitch_scale:.2f}.wav"
                    )
                    fpath = os.path.join(output_dir, fname)
                    # sf.write(fpath, audio_np, SAMPLE_RATE)
                    # print(f"\U0001f4be Saved \u2192 {fpath}")

                print("\n\U0001f3a7 Playback:")
                display(Audio(audio_np, rate=SAMPLE_RATE, autoplay=False))

            except Exception as e:
                print(f"\n\u274c Error: {e}")
                raise

    _btn.on_click(_synthesize)

    title = "NeMo \u00b7 tts_en_hifigan" if vocoder_type == "nemo" else "HiFiGAN VCTK v1"
    return widgets.VBox([
        widgets.HTML(f"<h3 style='margin:4px 0 10px 0'>\U0001f399\ufe0f TTS \u2014 {title}</h3>"),
        _w_text, _w_speaker, _w_pace, _w_pitch,
        widgets.HBox([_w_show_mel, _w_save]),
        _btn,
        widgets.HTML("<hr style='margin:10px 0'>"),
        _out,
    ], layout=widgets.Layout(
        padding="18px",
        border="1px solid #ddd",
        border_radius="8px",
        width="625px",
    ))


print("\u2705 build_tts_ui() factory defined — ready to use in Steps 5a and 5b")


✅ build_tts_ui() factory defined — ready to use in Steps 5a and 5b


---
## 🎛️ Step 5a — Synthesize with NeMo HiFiGAN
Requires Steps 1–4 to be run first.

In [14]:
display(build_tts_ui(
    fp_model     = fp_model,
    vocoder      = vocoder_nemo,
    vocoder_type = "nemo",
    device       = device,
    n_speakers   = N_SPEAKERS,
    output_dir   = OUTPUT_DIR,
))


---
## 🎛️ Step 5b — Synthesize with HiFiGAN VCTK v1
Requires Steps 1–4 to be run first.

In [15]:
display(build_tts_ui(
    fp_model     = fp_model,
    vocoder      = vocoder_vctk,
    vocoder_type = "vctk",
    device       = device,
    n_speakers   = N_SPEAKERS,
    output_dir   = OUTPUT_DIR,
))


---
## 🗂️ Step 6 — Batch Render All Speakers
Renders every speaker for a given sentence. Set `BATCH_VOCODER_TYPE` to switch vocoder.

---
## 📝 Reference

| Component | Details |
|---|---|
| **FastPitch** | NVIDIA DeepLearningExamples — your `.pt` checkpoint |
| **Weights** | `module.` prefix stripped, loaded `strict=True` |
| **NeMo HiFiGAN** | `tts_en_hifigan` via `HifiGanModel.from_pretrained()` |
| **VCTK HiFiGAN** | jik876/hifi-gan `generator_v1` checkpoint |
| **Speakers** | 10 speakers, IDs 0–9 |
| **Sample rate** | 22050 Hz |
| **Output files** | `{vocoder_type}_spk{id}_pace{p}_pitch{p}.wav` |
| **Tested** | Python 3.10 · NumPy 1.26.4 · PyTorch 2.6.0 |

### UI isolation guarantee
Both UIs are built by `build_tts_ui()`. Every widget, button, callback,
and tokenizer instance lives inside the function closure — running one
UI never touches the other's state.
Saved files are prefixed with the vocoder name so they never overwrite each other.
